# Death Count Extraction: Multi-Model Comparison

This notebook compares 6 different approaches for extracting death counts from conflict event descriptions:

1. **NT5-small** - Numerical reasoning specialist
2. **Flan-T5-base** - Instruction-following generalist
3. **IndicBART** - India-specific multilingual model
4. **Flan-T5-Large** - Large instruction-following generalist
5. **ConfliBERT-Poisson** - Regression baseline
6. **mT5-base** - Multilingual T5 model (100+ languages)
7. **Flan-T5-XL** - Large instruction-following generalist

**Dataset**: Armed Assault + Bombing incidents (~4,300 samples, 37% zeros, 63% with fatalities)

## Notebook Structure

1. **Setup & Configuration** - Environment setup and library imports
2. **Data Management** - Load, filter, and split data
3. **Model Training** - Train and evaluate each model
4. **Results & Analysis** - Compare models and analyze errors


## 1. Setup and Configuration


### 1.1 Environment Setup


#### 1.1.1 Colab Setup


In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 2) Clone or update the repo
BRANCH = "main"  # Change if working on a different branch

# Clone fresh each session 
!rm -rf /content/code-satp
!git clone -b $BRANCH --depth 1 https://github.com/eteitelbaum/code-satp.git /content/code-satp

# 3) Install dependencies from the cloned repo
%pip install -qU pip setuptools wheel
%pip install -r /content/code-satp/models/count-models/requirements.txt

# 4) Make result directories and add to path
import pathlib
import sys
pathlib.Path("/content/drive/MyDrive/colab/satp-results").mkdir(parents=True, exist_ok=True)
pathlib.Path("/content/drive/MyDrive/colab/satp-results/death-counts").mkdir(parents=True, exist_ok=True)
pathlib.Path("figures").mkdir(parents=True, exist_ok=True)  # For saving plots
sys.path.append("/content/code-satp/models/count-models")

# 5) GPU check
import torch
print("="*80)
print("✅ SETUP COMPLETE")
print("="*80)
print(f"📂 Cloned repo: /content/code-satp")
if torch.cuda.is_available():
    print(f"🖥️  GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  WARNING: Running on CPU! Training will be very slow.")
    print("   Go to Runtime > Change runtime type > Hardware accelerator > GPU")
print("="*80)

# 6) Set canonical task name for this notebook
TASK_NAME = "death-counts"


#### 1.1.2 Local Setup


In [ ]:
# # For local development, uncomment these lines:
# import sys
# import os

# # Add utils to path
# sys.path.append('./utils')

# # Local data and results paths
# DATA_PATH = "../../data/"
# RESULTS_PATH = "./results/"

# # Create local results directory
# os.makedirs(RESULTS_PATH, exist_ok=True)
# os.makedirs(os.path.join(RESULTS_PATH, "model_checkpoints"), exist_ok=True)
# os.makedirs("./figures", exist_ok=True)
# os.makedirs("./data", exist_ok=True)

# # Verify GPU availability
# import torch
# if torch.cuda.is_available():
#     print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
# else:
#     print("⚠️ GPU not available, using CPU")

# # Set task name
# TASK_NAME = "death-counts"

# print("✅ Local setup complete!")


### 1.2 Import Libraries


In [ ]:
# Core libraries
import os
import gc
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, mean_absolute_error

# Deep learning
import torch
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM,
    T5Tokenizer, T5ForConditionalGeneration,  # Explicit T5 classes for NT5 (per model card)
    Trainer, TrainingArguments, Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, default_data_collator,
    EarlyStoppingCallback
)
from datasets import Dataset

# Custom utilities
from utils import (
    extract_number, parse_prediction,
    compute_metrics, print_metrics,
    prepare_seq2seq_data, prepare_regression_data, prepare_qa_data,
    tokenize_seq2seq, tokenize_for_regression, tokenize_qa,
    PoissonRegressionModel,
    extract_qa_answer
)
from utils.training_utils import (
    create_seq2seq_training_args,
    create_qa_training_args,
    create_regression_training_args,
    cleanup_model
)
from utils.file_io import save_dataframe_csv, get_task_results_dir

# Configuration
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.style.use('seaborn-v0_8')

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")


## 2. Data Management


### 2.1 Load Raw Data


In [ ]:
# Try loading from cloned repository first (Colab) or fall back to local path
try:
    data_path = '/content/code-satp/data/satp_clean.csv'
    df = pd.read_csv(data_path)
    print("✅ Data loaded successfully from cloned repository")
except:
    print("🔧 Fallback: Loading from local path...")
    data_path = "../../data/satp_clean.csv"
    df = pd.read_csv(data_path)
    print("✅ Data loaded successfully from local path")

print(f"\nTotal incidents in full dataset: {len(df):,}")
print(f"Columns: {df.columns.tolist()[:10]}...")  # Show first 10 columns
df.head(3)


### 2.2 Filter to Armed Assault and Bombing Incidents


In [ ]:
# Filter to Armed Assault + Bombing incidents
violent_actions = ['Armed Assault', 'Bombing']
df_filtered = df[df['first_action'].isin(violent_actions)].copy()

# Ensure we always have a stable identifier for downstream merges
if 'incident_number' not in df_filtered.columns:
    df_filtered = df_filtered.reset_index(drop=True)
    df_filtered['incident_number'] = df_filtered.index.map(lambda i: f"generated_{i}")

# Keep only needed columns
cols_to_keep = ['incident_summary', 'total_fatalities', 'first_action', 'incident_number']
df_filtered = df_filtered[cols_to_keep]

# Remove rows with NA values in total_fatalities
df_filtered = df_filtered[df_filtered['total_fatalities'].notna()]
df_filtered = df_filtered[df_filtered['total_fatalities'] != 'NA']

# Convert to integer
df_filtered['total_fatalities'] = df_filtered['total_fatalities'].astype(int)

# Remove any rows with missing incident summaries
df_filtered = df_filtered[df_filtered['incident_summary'].notna()]
df_filtered = df_filtered[df_filtered['incident_summary'].str.strip() != '']

# Reset index while preserving identifiers
df_filtered = df_filtered.reset_index(drop=True)

print(f"\n{'='*60}")
print(f"Filtered to {len(df_filtered):,} incidents")
print(f"{'='*60}")

# Distribution summary
zero_count = (df_filtered['total_fatalities'] == 0).sum()
nonzero_count = (df_filtered['total_fatalities'] > 0).sum()
print(f"\nZero fatalities: {zero_count:,} ({zero_count/len(df_filtered)*100:.1f}%)")
print(f"With fatalities: {nonzero_count:,} ({nonzero_count/len(df_filtered)*100:.1f}%)")
print(f"Mean fatalities: {df_filtered['total_fatalities'].mean():.2f}")
print(f"Median fatalities: {df_filtered['total_fatalities'].median():.0f}")
print(f"Max fatalities: {df_filtered['total_fatalities'].max()}")

# By action type
print(f"\nBy action type:")
for action in violent_actions:
    action_df = df_filtered[df_filtered['first_action'] == action]
    zeros = (action_df['total_fatalities'] == 0).sum()
    print(f"  {action}: {len(action_df):,} incidents ({zeros/len(action_df)*100:.1f}% zeros)")


### 2.3 Visualize Fatality Distribution


In [ ]:
# Visualize fatality distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of all counts
counts = df_filtered['total_fatalities'].value_counts().sort_index()
axes[0].bar(counts.index[:20], counts.values[:20], color='steelblue', alpha=0.7)
axes[0].set_xlabel('Death Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Death Counts (0-19)')
axes[0].grid(axis='y', alpha=0.3)

# Binned distribution
bins = [0, 1, 2, 3, 5, 10, np.inf]
labels = ['0', '1', '2', '3-5', '6-10', '10+']
df_filtered['count_bin'] = pd.cut(df_filtered['total_fatalities'], bins=bins, labels=labels, right=False)
bin_counts = df_filtered['count_bin'].value_counts().sort_index()
axes[1].bar(range(len(bin_counts)), bin_counts.values, color='coral', alpha=0.7)
axes[1].set_xticks(range(len(bin_counts)))
axes[1].set_xticklabels(bin_counts.index, rotation=0)
axes[1].set_xlabel('Death Count Bin')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Binned Distribution')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/fatality_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nBinned distribution:")
for bin_label, count in bin_counts.items():
    print(f"  {bin_label}: {count:,} ({count/len(df_filtered)*100:.1f}%)")


### 2.4 Create Train/Validation/Test Splits

Using stratified splitting based on binned fatality counts to maintain class balance.


In [ ]:
# Create bins for stratification
# Using finer bins for better stratification
stratify_bins = pd.cut(df_filtered['total_fatalities'], 
                       bins=[0, 1, 2, 3, 6, np.inf], 
                       labels=['0', '1', '2', '3-5', '6+'],
                       right=False)

# First split: 80% train+val, 20% test
train_val_df, test_df = train_test_split(
    df_filtered,
    test_size=0.2,
    random_state=SEED,
    stratify=stratify_bins
)

# Second split: 75% train, 25% val (of the 80%)
# This gives us final split of 60% train, 20% val, 20% test
train_bins = pd.cut(train_val_df['total_fatalities'], 
                    bins=[0, 1, 2, 3, 6, np.inf], 
                    labels=['0', '1', '2', '3-5', '6+'],
                    right=False)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.25,  # 0.25 of 80% = 20% of total
    random_state=SEED,
    stratify=train_bins
)

print(f"Dataset splits:")
print(f"  Train: {len(train_df):,} ({len(train_df)/len(df_filtered)*100:.1f}%)")
print(f"  Val:   {len(val_df):,} ({len(val_df)/len(df_filtered)*100:.1f}%)")
print(f"  Test:  {len(test_df):,} ({len(test_df)/len(df_filtered)*100:.1f}%)")

# Check class balance
print(f"\nClass balance check:")
for name, df_split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    zero_pct = (df_split['total_fatalities'] == 0).mean() * 100
    mean_val = df_split['total_fatalities'].mean()
    print(f"  {name}: {zero_pct:.1f}% zeros, mean={mean_val:.2f}")


### 2.5 Save Data Splits


In [ ]:
# Reorder columns to keep identifiers up front before saving
cols_to_save = ['incident_number', 'incident_summary', 'total_fatalities']

train_df = train_df[cols_to_save]
val_df = val_df[cols_to_save]
test_df = test_df[cols_to_save]

# Save splits to disk using file_io utilities
save_dataframe_csv(train_df, 'train.csv', task_name=TASK_NAME)
save_dataframe_csv(val_df, 'val.csv', task_name=TASK_NAME)
save_dataframe_csv(test_df, 'test.csv', task_name=TASK_NAME)
save_dataframe_csv(df_filtered, 'armed_assault_bombing.csv', task_name=TASK_NAME)

print("✅ Saved data splits:")
print(f"  📁 {get_task_results_dir(TASK_NAME)}/train.csv")
print(f"  📁 {get_task_results_dir(TASK_NAME)}/val.csv")
print(f"  📁 {get_task_results_dir(TASK_NAME)}/test.csv")
print(f"  📁 {get_task_results_dir(TASK_NAME)}/armed_assault_bombing.csv")


### 2.6 Display Example Incidents


In [ ]:
# Show some examples from each split
print("\nExample incidents:")
print("\n" + "="*80)
for i, row in train_df.head(3).iterrows():
    summary = row['incident_summary'][:150] + "..." if len(row['incident_summary']) > 150 else row['incident_summary']
    print(f"Deaths: {row['total_fatalities']}")
    print(f"Text: {summary}")
    print("-" * 80)


## 3. Model Setup


### 3.1 Test Utility Functions

Verify that extraction and metrics utilities are working correctly.


In [ ]:
# Test extraction utilities (imported from utils/)
test_outputs = ["5", "5.", "The death count is 12", "approximately 7", "unknown", "30", ""]
print("Testing number extraction:")
for output in test_outputs:
    extracted = extract_number(output)
    print(f"  '{output}' → {extracted}")

# Test metrics utilities (imported from utils/)
test_preds = np.array([0, 1, 2, 5, 10])
test_labels = np.array([0, 1, 3, 5, 8])
test_metrics = compute_metrics(test_preds, test_labels)
print_metrics(test_metrics, "Test")


### 3.2 Training Configuration


In [ ]:
# Storage for results
all_results = {}
all_predictions = {}

# Training arguments
BATCH_SIZE = 8
MAX_EPOCHS = 10  # higher to allow model to converge
LEARNING_RATE_SEQ2SEQ = 3e-5
LEARNING_RATE_ENCODER = 2e-5
GENERATION_MAX_LENGTH = 16  # Max tokens for generated outputs (numbers only; accommodates up to 999 with buffer for occasional mistakes)

print("Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Max epochs: {MAX_EPOCHS} (early stopping enabled with patience=2)")
print(f"  LR (seq2seq): {LEARNING_RATE_SEQ2SEQ}")
print(f"  LR (encoder): {LEARNING_RATE_ENCODER}")
print(f"  Generation max length: {GENERATION_MAX_LENGTH}")


## 4. Model Training and Evaluation

Train and evaluate each of the 6 models sequentially. Helper functions for data preparation, tokenization, metrics, and model architectures are imported from the `utils/` module.

### 4.1 Model 1: NT5-small (Numerical Reasoning Specialist)


#### 4.1.1 Prepare Data for NT5


In [ ]:
print("\n" + "="*80)
print("NT5-SMALL: Numerical Reasoning Specialist")
print("="*80)

# Prepare datasets using utils function
nt5_train_data = prepare_seq2seq_data(train_df, model_type='nt5')
nt5_val_data = prepare_seq2seq_data(val_df, model_type='nt5')
nt5_test_data = prepare_seq2seq_data(test_df, model_type='nt5')

print(f"Training samples: {len(nt5_train_data['input'])}")
print(f"Validation samples: {len(nt5_val_data['input'])}")
print(f"Test samples: {len(nt5_test_data['input'])}")
print("\nExample input:")
print(nt5_train_data['input'][0][:200] + "...")
print(f"\nExample target: {nt5_train_data['target'][0]}")

#### 4.1.2 Load Model and Tokenizer


In [ ]:
# Load tokenizer and model
# Note: Using explicit T5 classes as shown in model card (https://huggingface.co/nielsr/nt5-small-rc1)
# The model card uses T5Tokenizer and T5ForConditionalGeneration, not Auto classes
model_name = "nielsr/nt5-small-rc1"
print(f"Loading {model_name}...")

# Use explicit T5 classes as per model card example
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Model loaded on {device}")
print(f"Model parameters: {model.num_parameters():,}")
print(f"Tokenizer type: {type(tokenizer).__name__}")
print(f"Model type: {type(model).__name__}")


#### 4.1.3 Tokenize Datasets

In [ ]:
# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_dict(nt5_train_data)
val_dataset = Dataset.from_dict(nt5_val_data)
test_dataset = Dataset.from_dict(nt5_test_data)

# Tokenize using utility function
train_dataset = train_dataset.map(
    lambda x: tokenize_seq2seq(x, tokenizer),
    batched=True,
    remove_columns=['input', 'target']
)

val_dataset = val_dataset.map(
    lambda x: tokenize_seq2seq(x, tokenizer),
    batched=True,
    remove_columns=['input', 'target']
)

test_dataset = test_dataset.map(
    lambda x: tokenize_seq2seq(x, tokenizer),
    batched=True,
    remove_columns=['input', 'target']
)

print("Datasets tokenized")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val: {len(val_dataset)} samples")
print(f"  Test: {len(test_dataset)} samples")


#### 4.1.4 Configure Training Arguments


In [ ]:
# Setup data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Training arguments using utility function
training_args = create_seq2seq_training_args(
    output_dir="results/model_checkpoints/nt5-small",
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE_SEQ2SEQ,
    num_epochs=MAX_EPOCHS,
    seed=SEED,
    generation_max_length=GENERATION_MAX_LENGTH
)

# Initialize trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Trainer configured")
print(f"  Output dir: {training_args.output_dir}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")


#### 4.1.5 Train the Model


In [ ]:
# Train the model
print("Training NT5-small...")
trainer.train()
print("Training complete!")

#### 4.1.6 Generate Predictions and Evaluate


In [ ]:
# Generate predictions on test set
print("Generating predictions on test set...")

predictions = trainer.predict(test_dataset)
predicted_ids = predictions.predictions

# Debug: Check what we're getting
print(f"\nDebug info:")
print(f"  Predicted IDs shape: {predicted_ids.shape}")
print(f"  First 5 predicted ID sequences: {predicted_ids[:5].tolist()}")
print(f"  First 5 decoded (skip_special_tokens=False): {[tokenizer.decode(ids, skip_special_tokens=False) for ids in predicted_ids[:5]]}")

# Decode predictions
# ⚠️ CRITICAL: T5 tokenizer removes numbers when skip_special_tokens=True!
# We need to decode with skip_special_tokens=False, then manually remove only </s> and <pad> tokens
decoded_preds = tokenizer.batch_decode(predicted_ids, skip_special_tokens=False)
# Manually clean up: remove </s> and <pad> but keep numbers
decoded_preds = [pred.replace('</s>', '').replace('<pad>', '').strip() for pred in decoded_preds]

print(f"\n  First 5 decoded (after manual cleanup): {decoded_preds[:5]}")
print(f"  Lengths of decoded strings: {[len(p) for p in decoded_preds[:5]]}")

# Extract numbers from predictions
nt5_predictions = []
extraction_successes = []
for i, pred in enumerate(decoded_preds):
    num = extract_number(pred)
    nt5_predictions.append(num if num is not None else 0)
    extraction_successes.append(num is not None)
    if i < 10:  # Show first 10 for debugging
        print(f"  Sample {i}: Raw='{pred}' | Extracted={num} | Success={num is not None}")

# Get true labels
true_labels = test_df['total_fatalities'].values

# Compute metrics
nt5_metrics = compute_metrics(nt5_predictions, true_labels, extraction_successes)
print_metrics(nt5_metrics, "NT5-small")

# Store results
all_results['nt5-small'] = nt5_metrics
prediction_dict = {
    'predictions': nt5_predictions,
    'raw_outputs': decoded_preds,
    'labels': true_labels,
    'incident_number': test_df['incident_number'].values,
    'incident_summary': test_df['incident_summary'].values,
}
all_predictions['nt5-small'] = prediction_dict

# Show some examples
print("\nExample predictions:")
for i in range(5):
    print(f"  True: {true_labels[i]}, Pred: {nt5_predictions[i]}, Raw: '{decoded_preds[i]}'")


#### 4.1.7 Save NT5 Results

In [ ]:
# Save NT5 results immediately
model_name = 'nt5-small'

# Save metrics to JSON
metrics_path = get_task_results_dir(TASK_NAME) / f"death_counts_{model_name}_metrics.json"
with open(metrics_path, 'w') as f:
    json.dump(nt5_metrics, f, indent=2)
print(f"✅ Saved metrics to {metrics_path}")

# Save predictions to CSV (including incident_number for merging with other models)
predictions_df = pd.DataFrame({
    'incident_number': test_df['incident_number'].values,
    'incident_summary': test_df['incident_summary'].values,
    'true_label': true_labels,
    'prediction': nt5_predictions,
    'raw_output': decoded_preds
})
save_dataframe_csv(predictions_df, f'death_counts_{model_name}_predictions.csv', task_name=TASK_NAME)
print(f"✅ Saved predictions to {get_task_results_dir(TASK_NAME)}/death_counts_{model_name}_predictions.csv")


#### 4.1.8 Save Model and Clean Up GPU Memory


In [ ]:
# Save the model and tokenizer to persistent location
model_name = 'nt5-small'
try:
    results_dir = get_task_results_dir(TASK_NAME)
    model_dir = results_dir / f"{model_name}_finetuned_model"
    model.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)
    print(f"✅ Model and tokenizer saved to: {model_dir}")
except NameError:
    # Fallback to Drive path for Colab
    model_path = f'/content/drive/MyDrive/colab/satp-results/death-counts/{model_name}_finetuned_model'
    import os
    os.makedirs(model_path, exist_ok=True)
    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    print(f"✅ Model and tokenizer saved to: {model_path}")

# Clear GPU memory using utility function
cleanup_model(model, trainer, tokenizer)
print(f"\n{model_name}: GPU memory cleared")


### 4.2 Model 2: Flan-T5-base (Instruction-Following)


#### 4.2.1 Prepare Data for Flan-T5


In [ ]:
print("\n" + "="*80)
print("FLAN-T5-BASE: Instruction-Following Generalist")
print("="*80)

# Prepare data using utils function
flant5_train_data = prepare_seq2seq_data(train_df, model_type='flan-t5')
flant5_val_data = prepare_seq2seq_data(val_df, model_type='flan-t5')
flant5_test_data = prepare_seq2seq_data(test_df, model_type='flan-t5')

print(f"Training samples: {len(flant5_train_data['input'])}")
print(f"Validation samples: {len(flant5_val_data['input'])}")
print(f"Test samples: {len(flant5_test_data['input'])}")
print("\nExample input:")
print(flant5_train_data['input'][0][:200] + "...")
print(f"\nExample target: {flant5_train_data['target'][0]}")


#### 4.2.2 Load Model and Tokenizer


In [ ]:
# Load tokenizer and model
model_name = "google/flan-t5-base"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Move to GPU if available
model = model.to(device)
print(f"Model loaded on {device}")
print(f"Model parameters: {model.num_parameters():,}")


#### 4.2.3 Tokenize Datasets

In [ ]:
# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_dict(flant5_train_data)
val_dataset = Dataset.from_dict(flant5_val_data)
test_dataset = Dataset.from_dict(flant5_test_data)

# Tokenize using utility function
train_dataset = train_dataset.map(
    lambda x: tokenize_seq2seq(x, tokenizer),
    batched=True,
    remove_columns=['input', 'target']
)

val_dataset = val_dataset.map(
    lambda x: tokenize_seq2seq(x, tokenizer),
    batched=True,
    remove_columns=['input', 'target']
)

test_dataset = test_dataset.map(
    lambda x: tokenize_seq2seq(x, tokenizer),
    batched=True,
    remove_columns=['input', 'target']
)

print("Datasets tokenized")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val: {len(val_dataset)} samples")
print(f"  Test: {len(test_dataset)} samples")


#### 4.2.4 Configure Training Arguments


In [ ]:
# Setup data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Training arguments using utility function
training_args = create_seq2seq_training_args(
    output_dir="results/model_checkpoints/flan-t5-base",
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE_SEQ2SEQ,
    num_epochs=MAX_EPOCHS,
    seed=SEED,
    generation_max_length=GENERATION_MAX_LENGTH
)

# Initialize trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Trainer configured")
print(f"  Output dir: {training_args.output_dir}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")


#### 4.2.5 Train the Model


In [ ]:
# Train the model
print("Training Flan-T5-base...")
trainer.train()
print("Training complete!")


#### 4.2.6 Generate Predictions and Evaluate

In [ ]:
# Generate predictions on test set
print("Generating predictions on test set...")

predictions = trainer.predict(test_dataset)
predicted_ids = predictions.predictions

# Decode predictions
decoded_preds = tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)

# Extract numbers from predictions
flant5_predictions = []
extraction_successes = []
for pred in decoded_preds:
    num = extract_number(pred)
    flant5_predictions.append(num if num is not None else 0)
    extraction_successes.append(num is not None)

# Get true labels
true_labels = test_df['total_fatalities'].values

# Compute metrics
flant5_metrics = compute_metrics(flant5_predictions, true_labels, extraction_successes)
print_metrics(flant5_metrics, "Flan-T5-base")

# Store results
all_results['flan-t5-base'] = flant5_metrics
prediction_dict = {
    'predictions': flant5_predictions,
    'raw_outputs': decoded_preds,
    'labels': true_labels,
    'incident_number': test_df['incident_number'].values,
    'incident_summary': test_df['incident_summary'].values,
}
all_predictions['flan-t5-base'] = prediction_dict

# Show some examples
print("\nExample predictions:")
for i in range(5):
    print(f"  True: {true_labels[i]}, Pred: {flant5_predictions[i]}, Raw: '{decoded_preds[i]}'")


#### 4.2.7 Save Flan-T5 Results


In [ ]:
# Save Flan-T5 results immediately
model_name = 'flan-t5-base'

# Save metrics to JSON
metrics_path = get_task_results_dir(TASK_NAME) / f"death_counts_{model_name}_metrics.json"
with open(metrics_path, 'w') as f:
    json.dump(flant5_metrics, f, indent=2)
print(f"✅ Saved metrics to {metrics_path}")

# Save predictions to CSV (including incident_number for merging with other models)
predictions_df = pd.DataFrame({
    'incident_number': test_df['incident_number'].values,
    'incident_summary': test_df['incident_summary'].values,
    'true_label': true_labels,
    'prediction': flant5_predictions,
    'raw_output': decoded_preds
})
save_dataframe_csv(predictions_df, f'death_counts_{model_name}_predictions.csv', task_name=TASK_NAME)
print(f"✅ Saved predictions to {get_task_results_dir(TASK_NAME)}/death_counts_{model_name}_predictions.csv")


#### 4.2.8 Save Model and Clean Up GPU Memory


In [ ]:
# Save the model and tokenizer to persistent location
model_name = 'flan-t5-base'
try:
    results_dir = get_task_results_dir(TASK_NAME)
    model_dir = results_dir / f"{model_name}_finetuned_model"
    model.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)
    print(f"✅ Model and tokenizer saved to: {model_dir}")
except NameError:
    # Fallback to Drive path for Colab
    model_path = f'/content/drive/MyDrive/colab/satp-results/death-counts/{model_name}_finetuned_model'
    import os
    os.makedirs(model_path, exist_ok=True)
    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    print(f"✅ Model and tokenizer saved to: {model_path}")

# Clear GPU memory using utility function
cleanup_model(model, trainer, tokenizer)
print(f"\n{model_name}: GPU memory cleared")


### 4.3 Model 3: IndicBART (India-Specific Multilingual)


#### 4.3.1 Prepare Data for IndicBART

In [ ]:
print("\n" + "="*80)
print("INDICBART: India-Specific Multilingual Model")
print("="*80)

# Prepare data using utils function (same format as Flan-T5)
indicbart_train_data = prepare_seq2seq_data(train_df, model_type='flan-t5')
indicbart_val_data = prepare_seq2seq_data(val_df, model_type='flan-t5')
indicbart_test_data = prepare_seq2seq_data(test_df, model_type='flan-t5')

print(f"Training samples: {len(indicbart_train_data['input'])}")
print(f"Validation samples: {len(indicbart_val_data['input'])}")
print(f"Test samples: {len(indicbart_test_data['input'])}")
print("\nExample input:")
print(indicbart_train_data['input'][0][:200] + "...")
print(f"\nExample target: {indicbart_train_data['target'][0]}")


#### 4.3.2 Load Model and Tokenizer


In [ ]:
# Load tokenizer and model
model_name = "ai4bharat/IndicBARTSS"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Move to GPU if available
model = model.to(device)
print(f"Model loaded on {device}")
print(f"Model parameters: {model.num_parameters():,}")


#### 4.3.3 Tokenize Datasets


In [ ]:
# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_dict(indicbart_train_data)
val_dataset = Dataset.from_dict(indicbart_val_data)
test_dataset = Dataset.from_dict(indicbart_test_data)

# Tokenize using utility function
train_dataset = train_dataset.map(
    lambda x: tokenize_seq2seq(x, tokenizer),
    batched=True,
    remove_columns=['input', 'target']
)

val_dataset = val_dataset.map(
    lambda x: tokenize_seq2seq(x, tokenizer),
    batched=True,
    remove_columns=['input', 'target']
)

test_dataset = test_dataset.map(
    lambda x: tokenize_seq2seq(x, tokenizer),
    batched=True,
    remove_columns=['input', 'target']
)

print("Datasets tokenized")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val: {len(val_dataset)} samples")
print(f"  Test: {len(test_dataset)} samples")


#### 4.3.4 Configure Training Arguments


In [ ]:
# Setup data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Training arguments using utility function
training_args = create_seq2seq_training_args(
    output_dir="results/model_checkpoints/indicbart",
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE_SEQ2SEQ,
    num_epochs=MAX_EPOCHS,
    seed=SEED,
    generation_max_length=GENERATION_MAX_LENGTH
)

# Initialize trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Trainer configured")
print(f"  Output dir: {training_args.output_dir}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")


#### 4.3.5 Train the Model


In [ ]:
# Train the model
print("Training IndicBART...")
trainer.train()
print("Training complete!")


#### 4.3.6 Generate Predictions and Evaluate


In [ ]:
# Generate predictions on test set
print("Generating predictions on test set...")

predictions = trainer.predict(test_dataset)
predicted_ids = predictions.predictions

# Decode predictions
decoded_preds = tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)

# Extract numbers from predictions
indicbart_predictions = []
extraction_successes = []
for pred in decoded_preds:
    num = extract_number(pred)
    indicbart_predictions.append(num if num is not None else 0)
    extraction_successes.append(num is not None)

# Get true labels
true_labels = test_df['total_fatalities'].values

# Compute metrics
indicbart_metrics = compute_metrics(indicbart_predictions, true_labels, extraction_successes)
print_metrics(indicbart_metrics, "IndicBART")

# Store results
all_results['indicbart'] = indicbart_metrics
prediction_dict = {
    'predictions': indicbart_predictions,
    'raw_outputs': decoded_preds,
    'labels': true_labels,
    'incident_number': test_df['incident_number'].values,
    'incident_summary': test_df['incident_summary'].values,
}
all_predictions['indicbart'] = prediction_dict

# Show some examples
print("\nExample predictions:")
for i in range(5):
    print(f"  True: {true_labels[i]}, Pred: {indicbart_predictions[i]}, Raw: '{decoded_preds[i]}'")


#### 4.3.7 Save IndicBART Results


In [ ]:
# Save IndicBART results immediately
model_name = 'indicbart'

# Save metrics to JSON
metrics_path = get_task_results_dir(TASK_NAME) / f"death_counts_{model_name}_metrics.json"
with open(metrics_path, 'w') as f:
    json.dump(indicbart_metrics, f, indent=2)
print(f"✅ Saved metrics to {metrics_path}")

# Save predictions to CSV (including incident_number for merging with other models)
predictions_df = pd.DataFrame({
    'incident_number': test_df['incident_number'].values,
    'incident_summary': test_df['incident_summary'].values,
    'true_label': true_labels,
    'prediction': indicbart_predictions,
    'raw_output': decoded_preds
})
save_dataframe_csv(predictions_df, f'death_counts_{model_name}_predictions.csv', task_name=TASK_NAME)
print(f"✅ Saved predictions to {get_task_results_dir(TASK_NAME)}/death_counts_{model_name}_predictions.csv")


#### 4.3.8 Save Model and Clean Up GPU Memory


In [ ]:
# Save the model and tokenizer to persistent location
model_name = 'indicbart'
try:
    results_dir = get_task_results_dir(TASK_NAME)
    model_dir = results_dir / f"{model_name}_finetuned_model"
    model.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)
    print(f"✅ Model and tokenizer saved to: {model_dir}")
except NameError:
    # Fallback to Drive path for Colab
    model_path = f'/content/drive/MyDrive/colab/satp-results/death-counts/{model_name}_finetuned_model'
    import os
    os.makedirs(model_path, exist_ok=True)
    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    print(f"✅ Model and tokenizer saved to: {model_path}")

# Clear GPU memory using utility function
cleanup_model(model, trainer, tokenizer)
print(f"\n{model_name}: GPU memory cleared")


### 4.4 Model 4: Flan-T5-Large (Large Instruction-Following)

#### 4.4.1 Prepare Data for Flan-T5-Large

In [ ]:
print("\n" + "="*80)
print("FLAN-T5-LARGE: Large Instruction-Following Generalist")
print("="*80)

# Prepare data using utils function (same format as Flan-T5-base)
flant5l_train_data = prepare_seq2seq_data(train_df, model_type='flan-t5')
flant5l_val_data = prepare_seq2seq_data(val_df, model_type='flan-t5')
flant5l_test_data = prepare_seq2seq_data(test_df, model_type='flan-t5')

print(f"Training samples: {len(flant5l_train_data['input'])}")
print(f"Validation samples: {len(flant5l_val_data['input'])}")
print(f"Test samples: {len(flant5l_test_data['input'])}")
print("\nExample input:")
print(flant5l_train_data['input'][0][:200] + "...")
print(f"\nExample target: {flant5l_train_data['target'][0]}")


#### 4.4.2 Load Model and Tokenizer

In [ ]:
# Load tokenizer and model
model_name = "google/flan-t5-large"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Move to GPU if available
model = model.to(device)
print(f"Model loaded on {device}")
print(f"Model parameters: {model.num_parameters():,}")


#### 4.4.3 Tokenize Datasets


In [ ]:
# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_dict(flant5l_train_data)
val_dataset = Dataset.from_dict(flant5l_val_data)
test_dataset = Dataset.from_dict(flant5l_test_data)

# Tokenize using utility function
train_dataset = train_dataset.map(
    lambda x: tokenize_seq2seq(x, tokenizer),
    batched=True,
    remove_columns=['input', 'target']
)

val_dataset = val_dataset.map(
    lambda x: tokenize_seq2seq(x, tokenizer),
    batched=True,
    remove_columns=['input', 'target']
)

test_dataset = test_dataset.map(
    lambda x: tokenize_seq2seq(x, tokenizer),
    batched=True,
    remove_columns=['input', 'target']
)

print("Datasets tokenized")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val: {len(val_dataset)} samples")
print(f"  Test: {len(test_dataset)} samples")


#### 4.4.4 Configure Training Arguments


In [ ]:
# Setup data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Training arguments using utility function
training_args = create_seq2seq_training_args(
    output_dir="results/model_checkpoints/flan-t5-large",
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE_SEQ2SEQ,
    num_epochs=MAX_EPOCHS,
    seed=SEED,
    generation_max_length=GENERATION_MAX_LENGTH
)

# Initialize trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Trainer configured")
print(f"  Output dir: {training_args.output_dir}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")


#### 4.4.5 Train the Model


In [ ]:
# Train the model
print("Training Flan-T5-Large...")
trainer.train()
print("Training complete!")


#### 4.4.6 Generate Predictions and Evaluate


In [ ]:
# Generate predictions on test set
print("Generating predictions on test set...")

predictions = trainer.predict(test_dataset)
predicted_ids = predictions.predictions

# Decode predictions
decoded_preds = tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)

# Extract numbers from predictions
flant5l_predictions = []
extraction_successes = []
for pred in decoded_preds:
    num = extract_number(pred)
    flant5l_predictions.append(num if num is not None else 0)
    extraction_successes.append(num is not None)

# Get true labels
true_labels = test_df['total_fatalities'].values

# Compute metrics
flant5l_metrics = compute_metrics(flant5l_predictions, true_labels, extraction_successes)
print_metrics(flant5l_metrics, "Flan-T5-Large")

# Store results
all_results['flan-t5-large'] = flant5l_metrics
prediction_dict = {
    'predictions': flant5l_predictions,
    'raw_outputs': decoded_preds,
    'labels': true_labels,
    'incident_number': test_df['incident_number'].values,
    'incident_summary': test_df['incident_summary'].values,
}
all_predictions['flan-t5-large'] = prediction_dict

# Show some examples
print("\nExample predictions:")
for i in range(5):
    print(f"  True: {true_labels[i]}, Pred: {flant5l_predictions[i]}, Raw: '{decoded_preds[i]}'")


#### 4.4.7 Save Flan-T5-Large Results


In [ ]:
# Save Flan-T5-Large results immediately
model_name = 'flan-t5-large'

# Save metrics to JSON
metrics_path = get_task_results_dir(TASK_NAME) / f"death_counts_{model_name}_metrics.json"
with open(metrics_path, 'w') as f:
    json.dump(flant5l_metrics, f, indent=2)
print(f"✅ Saved metrics to {metrics_path}")

# Save predictions to CSV (including incident_number for merging with other models)
predictions_df = pd.DataFrame({
    'incident_number': test_df['incident_number'].values,
    'incident_summary': test_df['incident_summary'].values,
    'true_label': true_labels,
    'prediction': flant5l_predictions,
    'raw_output': decoded_preds
})
save_dataframe_csv(predictions_df, f'death_counts_{model_name}_predictions.csv', task_name=TASK_NAME)
print(f"✅ Saved predictions to {get_task_results_dir(TASK_NAME)}/death_counts_{model_name}_predictions.csv")


#### 4.4.8 Save Model and Clean Up GPU Memory


In [ ]:
# Save the model and tokenizer to persistent location
model_name = 'flan-t5-large'
try:
    results_dir = get_task_results_dir(TASK_NAME)
    model_dir = results_dir / f"{model_name}_finetuned_model"
    model.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)
    print(f"✅ Model and tokenizer saved to: {model_dir}")
except NameError:
    # Fallback to Drive path for Colab
    model_path = f'/content/drive/MyDrive/colab/satp-results/death-counts/{model_name}_finetuned_model'
    import os
    os.makedirs(model_path, exist_ok=True)
    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    print(f"✅ Model and tokenizer saved to: {model_path}")

# Clear GPU memory using utility function
cleanup_model(model, trainer, tokenizer)
print(f"\n{model_name}: GPU memory cleared")


### 4.5 Model 5: ConfliBERT-Poisson (Regression Baseline)


#### 4.5.1 Prepare Data for Regression

In [ ]:
print("\n" + "="*80)
print("CONFLIBERT-POISSON: Regression Baseline (Domain-Specific)")
print("="*80)

# Prepare data for regression using utility function
regression_train_data = prepare_regression_data(train_df)
regression_val_data = prepare_regression_data(val_df)
regression_test_data = prepare_regression_data(test_df)

print(f"Training samples: {len(regression_train_data['text'])}")
print(f"Validation samples: {len(regression_val_data['text'])}")
print(f"Test samples: {len(regression_test_data['text'])}")
print(f"\nExample text:")
print(regression_train_data['text'][0][:200] + "...")
print(f"Example label: {regression_train_data['labels'][0]}")


#### 4.5.2 Load Model and Tokenizer


In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("snowood1/ConfliBERT-scr-cased")

# Load Poisson regression model from utils
print("Loading ConfliBERT-Poisson...")
model = PoissonRegressionModel("snowood1/ConfliBERT-scr-cased")

# Move to GPU if available
model = model.to(device)
print(f"Model loaded on {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


#### 4.5.3 Tokenize Datasets

In [ ]:
# Convert to HuggingFace Dataset format
regression_train = Dataset.from_dict(regression_train_data)
regression_val = Dataset.from_dict(regression_val_data)
regression_test = Dataset.from_dict(regression_test_data)

# Tokenize using utility function
regression_train = regression_train.map(
    lambda x: tokenize_for_regression(x, tokenizer),
    batched=True
)

regression_val = regression_val.map(
    lambda x: tokenize_for_regression(x, tokenizer),
    batched=True
)

regression_test = regression_test.map(
    lambda x: tokenize_for_regression(x, tokenizer),
    batched=True
)

print("Datasets tokenized")
print(f"  Train: {len(regression_train)} samples")
print(f"  Val: {len(regression_val)} samples")
print(f"  Test: {len(regression_test)} samples")


#### 4.5.4 Configure Training Arguments


In [ ]:
# Training arguments using utility function
training_args = create_regression_training_args(
       output_dir="results/model_checkpoints/conflibert-poisson",
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE_ENCODER,
    num_epochs=MAX_EPOCHS,
    seed=SEED
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=regression_train,
    eval_dataset=regression_val,
    data_collator=default_data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Trainer configured")
print(f"  Output dir: {training_args.output_dir}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")


#### 4.5.5 Train the Model


In [ ]:
# Train the model
print("Training ConfliBERT-Poisson...")
trainer.train()
print("Training complete!")


#### 4.5.6 Generate Predictions and Evaluate

In [ ]:
# Generate predictions on test set
print("Generating predictions on test set...")

predictions = trainer.predict(regression_test)
raw_preds = predictions.predictions

# Round and floor at 0
conflibert_predictions = [max(0, round(p)) for p in raw_preds]

# Get true labels
true_labels = test_df['total_fatalities'].values

# Compute metrics
conflibert_metrics = compute_metrics(conflibert_predictions, true_labels)
print_metrics(conflibert_metrics, "ConfliBERT-Poisson")

# Store results
all_results['conflibert-poisson'] = conflibert_metrics
prediction_dict = {
    'predictions': conflibert_predictions,
    'raw_outputs': raw_preds,
    'labels': true_labels,
    'incident_number': test_df['incident_number'].values,
    'incident_summary': test_df['incident_summary'].values,
}
all_predictions['conflibert-poisson'] = prediction_dict

# Show some examples
print("\nExample predictions:")
for i in range(5):
    print(f"  True: {true_labels[i]}, Pred: {conflibert_predictions[i]}, Raw: {raw_preds[i]:.2f}")


#### 4.5.7 Save ConfliBERT-Poisson Results


In [ ]:
# Save ConfliBERT-Poisson results immediately
model_name = 'conflibert-poisson'

# Save metrics to JSON
metrics_path = get_task_results_dir(TASK_NAME) / f"death_counts_{model_name}_metrics.json"
with open(metrics_path, 'w') as f:
    json.dump(conflibert_metrics, f, indent=2)
print(f"✅ Saved metrics to {metrics_path}")

# Save predictions to CSV (including incident_number for merging with other models)
predictions_df = pd.DataFrame({
    'incident_number': test_df['incident_number'].values,
    'incident_summary': test_df['incident_summary'].values,
    'true_label': true_labels,
    'prediction': conflibert_predictions,
    'raw_output': [str(p) for p in raw_preds]  # Convert to strings for CSV
})
save_dataframe_csv(predictions_df, f'death_counts_{model_name}_predictions.csv', task_name=TASK_NAME)
print(f"✅ Saved predictions to {get_task_results_dir(TASK_NAME)}/death_counts_{model_name}_predictions.csv")


#### 4.5.8 Save Model and Clean Up GPU Memory


### 4.6 Model 6: mT5-base (Multilingual T5)


#### 4.6.1 Prepare Data for mT5


In [ ]:
print("\n" + "="*80)
print("MT5-BASE: Multilingual T5 Model (100+ Languages)")
print("="*80)

# Prepare data using utils function (mT5 requires language prefix)
mt5_train_data = prepare_seq2seq_data(train_df, model_type='mt5')
mt5_val_data = prepare_seq2seq_data(val_df, model_type='mt5')
mt5_test_data = prepare_seq2seq_data(test_df, model_type='mt5')

print(f"Training samples: {len(mt5_train_data['input'])}")
print(f"Validation samples: {len(mt5_val_data['input'])}")
print(f"Test samples: {len(mt5_test_data['input'])}")
print("\nExample input:")
print(mt5_train_data['input'][0][:200] + "...")
print(f"\nExample target: {mt5_train_data['target'][0]}")


#### 4.6.2 Load Model and Tokenizer


In [ ]:
# Load tokenizer and model
model_name = "google/mt5-base"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Move to GPU if available
model = model.to(device)
print(f"Model loaded on {device}")
print(f"Model parameters: {model.num_parameters():,}")


#### 4.6.3 Tokenize Datasets


In [ ]:
# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_dict(mt5_train_data)
val_dataset = Dataset.from_dict(mt5_val_data)
test_dataset = Dataset.from_dict(mt5_test_data)

# Tokenize using utility function
train_dataset = train_dataset.map(
    lambda x: tokenize_seq2seq(x, tokenizer),
    batched=True,
    remove_columns=['input', 'target']
)

val_dataset = val_dataset.map(
    lambda x: tokenize_seq2seq(x, tokenizer),
    batched=True,
    remove_columns=['input', 'target']
)

test_dataset = test_dataset.map(
    lambda x: tokenize_seq2seq(x, tokenizer),
    batched=True,
    remove_columns=['input', 'target']
)

print("Datasets tokenized")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val: {len(val_dataset)} samples")
print(f"  Test: {len(test_dataset)} samples")


#### 4.6.4 Configure Training Arguments


In [ ]:
# Setup data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Training arguments using utility function
training_args = create_seq2seq_training_args(
    output_dir="results/model_checkpoints/mt5-base",
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE_SEQ2SEQ,
    num_epochs=MAX_EPOCHS,
    seed=SEED,
    generation_max_length=GENERATION_MAX_LENGTH
)

# Initialize trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Trainer configured")
print(f"  Output dir: {training_args.output_dir}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")


#### 4.6.5 Train the Model


In [ ]:
# Train the model
print("Training mT5-base...")
trainer.train()
print("Training complete!")


#### 4.6.6 Generate Predictions and Evaluate


In [ ]:
# Generate predictions on test set
print("Generating predictions on test set...")

predictions = trainer.predict(test_dataset)
predicted_ids = predictions.predictions

# Decode predictions
decoded_preds = tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)

# Extract numbers from predictions
mt5_predictions = []
extraction_successes = []
for pred in decoded_preds:
    num = extract_number(pred)
    mt5_predictions.append(num if num is not None else 0)
    extraction_successes.append(num is not None)

# Get true labels
true_labels = test_df['total_fatalities'].values

# Compute metrics
mt5_metrics = compute_metrics(mt5_predictions, true_labels, extraction_successes)
print_metrics(mt5_metrics, "mT5-base")

# Store results
all_results['mt5-base'] = mt5_metrics
prediction_dict = {
    'predictions': mt5_predictions,
    'raw_outputs': decoded_preds,
    'labels': true_labels,
    'incident_number': test_df['incident_number'].values,
    'incident_summary': test_df['incident_summary'].values,
}
all_predictions['mt5-base'] = prediction_dict

# Show some examples
print("\nExample predictions:")
for i in range(5):
    print(f"  True: {true_labels[i]}, Pred: {mt5_predictions[i]}, Raw: '{decoded_preds[i]}'")


#### 4.6.7 Save mT5 Results


In [ ]:
# Save mT5 results immediately
model_name = 'mt5-base'

# Save metrics to JSON
metrics_path = get_task_results_dir(TASK_NAME) / f"death_counts_{model_name}_metrics.json"
with open(metrics_path, 'w') as f:
    json.dump(mt5_metrics, f, indent=2)
print(f"✅ Saved metrics to {metrics_path}")

# Save predictions to CSV (including incident_number for merging with other models)
predictions_df = pd.DataFrame({
    'incident_number': test_df['incident_number'].values,
    'incident_summary': test_df['incident_summary'].values,
    'true_label': true_labels,
    'prediction': mt5_predictions,
    'raw_output': decoded_preds
})
save_dataframe_csv(predictions_df, f'death_counts_{model_name}_predictions.csv', task_name=TASK_NAME)
print(f"✅ Saved predictions to {get_task_results_dir(TASK_NAME)}/death_counts_{model_name}_predictions.csv")


#### 4.6.8 Save Model and Clean Up GPU Memory


In [ ]:
# Save the model and tokenizer to persistent location
model_name = 'mt5-base'
try:
    results_dir = get_task_results_dir(TASK_NAME)
    model_dir = results_dir / f"{model_name}_finetuned_model"
    model.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)
    print(f"✅ Model and tokenizer saved to: {model_dir}")
except NameError:
    # Fallback to Drive path for Colab
    model_path = f'/content/drive/MyDrive/colab/satp-results/death-counts/{model_name}_finetuned_model'
    import os
    os.makedirs(model_path, exist_ok=True)
    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    print(f"✅ Model and tokenizer saved to: {model_path}")

# Clear GPU memory using utility function
cleanup_model(model, trainer, tokenizer)
print(f"\n{model_name}: GPU memory cleared")


### 4.7 Flan-T5-XL (QLoRA fine-tuning)

This section fine-tunes `google/flan-t5-xl` with QLoRA and saves metrics and predictions compatible with the existing comparison utilities.


#### 4.7.1 Install Dependencies

In [ ]:
# Install/verify dependencies for QLoRA (PEFT, bitsandbytes, accelerate)
try:
    import peft  # noqa: F401
    import bitsandbytes as bnb  # noqa: F401
    import accelerate  # noqa: F401
    print("PEFT/bitsandbytes/accelerate already installed")
except Exception:
    import sys, subprocess
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", "peft", "bitsandbytes", "accelerate"])  # noqa: S603
    except Exception:
        pass



#### 4.7.2 Prepare Data for Flan-T5-XL

In [ ]:
# Prepare data for Flan-T5-XL (reuse seq2seq format)
from utils import prepare_seq2seq_data, tokenize_seq2seq, compute_metrics, print_metrics
from datasets import Dataset

flanxl_train = prepare_seq2seq_data(train_df, model_type='flan-t5')
flanxl_val = prepare_seq2seq_data(val_df, model_type='flan-t5')
flanxl_test = prepare_seq2seq_data(test_df, model_type='flan-t5')

train_ds = Dataset.from_dict(flanxl_train)
val_ds = Dataset.from_dict(flanxl_val)
test_ds = Dataset.from_dict(flanxl_test)

print(f"Flan-T5-XL QLoRA: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")


#### 4.7.3 Load Model and Tokenizer

In [ ]:
# Tokenizer and model with QLoRA adapters
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

model_name = "google/flan-t5-xl"
print(f"Loading {model_name} with 4-bit quantization...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    device_map="auto",
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.enable_input_require_grads()

peft_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
    target_modules=["q", "k", "v", "o", "wi_0", "wi_1", "wo"],
)
model = get_peft_model(model, peft_cfg)
model.print_trainable_parameters()

print("Model prepared with LoRA adapters")


#### 4.7.4 Tokenize Datasets

In [ ]:
# Tokenize datasets
train_tok = train_ds.map(lambda x: tokenize_seq2seq(x, tokenizer, max_input_length=512, max_target_length=10), batched=True, remove_columns=['input','target'])
val_tok = val_ds.map(lambda x: tokenize_seq2seq(x, tokenizer, max_input_length=512, max_target_length=10), batched=True, remove_columns=['input','target'])
test_tok = test_ds.map(lambda x: tokenize_seq2seq(x, tokenizer, max_input_length=512, max_target_length=10), batched=True, remove_columns=['input','target'])

print("Tokenization complete")


#### 4.7.5 Configure Training Arguments

In [ ]:
# Training with Seq2SeqTrainer
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer
from utils.training_utils import create_seq2seq_training_args

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = create_seq2seq_training_args(
    output_dir="results/model_checkpoints/flan-t5-xl-lora",
    batch_size=2,
    learning_rate=5e-5,
    num_epochs=3,
    seed=SEED,
    generation_max_length=16,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Trainer configured for Flan-T5-XL QLoRA")


#### 4.7.6 Train the Model

In [ ]:
# Train (you can interrupt early if convergence is reached)
print("Training Flan-T5-XL (QLoRA)...")
trainer.train()
print("Training complete!")


#### 4.7.7 Generate Predictions and Evaluate

In [ ]:
# Generate predictions on test set and evaluate
from utils import extract_number

print("Generating predictions with Flan-T5-XL (QLoRA)...")
preds = trainer.predict(test_tok)
pred_ids = preds.predictions

# Decode and clean outputs
decoded = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)

# Extract numbers
pred_numbers = []
extraction_successes = []
for s in decoded:
    num = extract_number(s)
    pred_numbers.append(num if num is not None else 0)
    extraction_successes.append(num is not None)

true_labels = test_df['total_fatalities'].values
metrics = compute_metrics(pred_numbers, true_labels, extraction_successes)
print_metrics(metrics, "Flan-T5-XL-QLoRA")


#### 4.7.8 Save Metrics and Predictions

In [ ]:
# Save metrics and predictions
import json
import pandas as pd
from utils.file_io import get_task_results_dir

model_key = 'flan-t5-xl-lora'
results_dir = get_task_results_dir(TASK_NAME)

# Metrics JSON
metrics_path = results_dir / f"death_counts_{model_key}_metrics.json"
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"✅ Saved metrics to {metrics_path}")

# Predictions CSV
pred_df = pd.DataFrame({
    'incident_number': test_df['incident_number'].values,
    'incident_summary': test_df['incident_summary'].values,
    'true_label': true_labels,
    'prediction': pred_numbers,
    'raw_output': decoded,
})
preds_path = results_dir / f"death_counts_{model_key}_predictions.csv"
pred_df.to_csv(preds_path, index=False)
print(f"✅ Saved predictions to {preds_path}")


#### 4.7.9 Save LoRA Adapter

In [ ]:
# Save LoRA adapter
from pathlib import Path
adapter_dir = get_task_results_dir(TASK_NAME) / f"{model_key}_finetuned_model"
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"✅ LoRA adapter and tokenizer saved to: {adapter_dir}")


#### 4.7.10 Optional: Include in Comparison Tables

In [ ]:
# Optional: include Flan-T5-XL-QLoRA in comparison tables
from utils.file_io import get_task_results_dir
import pandas as pd

results_dir = get_task_results_dir(TASK_NAME)
metrics_path = results_dir / "death_counts_flan-t5-xl-lora_metrics.json"
preds_path = results_dir / "death_counts_flan-t5-xl-lora_predictions.csv"

if metrics_path.exists() and preds_path.exists():
    print("Flan-T5-XL-QLoRA results available; you can integrate them into your comparison section.")
else:
    print("Run the training/eval cells above to produce results before comparison.")

## 5. Results Comparison

Compare all models across multiple metrics.


### 5.1 Load Previously Saved Results

If any models were run previously and saved, load them here. This allows resuming after failures.


In [ ]:
# Load any previously saved results
print("Loading previously saved results...")
print("=" * 80)

results_dir = get_task_results_dir(TASK_NAME)
model_names = ['nt5-small', 'flan-t5-base', 'indicbart', 'flan-t5-large', 'conflibert-poisson', 'mt5-base']

loaded_count = 0
skipped_count = 0

for model_name in model_names:
    # Check if this model already has in-memory results
    if model_name in all_results and all_results[model_name] is not None:
        print(f"✓ {model_name}: Already in memory (from this session)")
        skipped_count += 1
        continue
    
    # Try to load from disk
    metrics_path = results_dir / f"death_counts_{model_name}_metrics.json"
    predictions_path = results_dir / f"death_counts_{model_name}_predictions.csv"
    
    if metrics_path.exists() and predictions_path.exists():
        try:
            # Load metrics
            with open(metrics_path, 'r') as f:
                metrics = json.load(f)
            
            # Load predictions
            preds_df = pd.read_csv(predictions_path)
            
            # Store in dictionaries
            all_results[model_name] = metrics
            prediction_dict = {
                'predictions': preds_df['prediction'].values,
                'raw_outputs': preds_df['raw_output'].values,
                'labels': preds_df['true_label'].values
            }
            # Preserve identifiers and context if present
            if 'incident_number' in preds_df.columns:
                prediction_dict['incident_number'] = preds_df['incident_number'].values
            elif 'incident_number' in test_df.columns and len(test_df) == len(preds_df):
                prediction_dict['incident_number'] = test_df['incident_number'].values

            if 'incident_summary' in preds_df.columns:
                prediction_dict['incident_summary'] = preds_df['incident_summary'].values
            elif 'incident_summary' in test_df.columns and len(test_df) == len(preds_df):
                prediction_dict['incident_summary'] = test_df['incident_summary'].values

            all_predictions[model_name] = prediction_dict
            
            print(f"✓ {model_name}: Loaded from disk")
            loaded_count += 1
        except Exception as e:
            print(f"✗ {model_name}: Error loading - {e}")
    else:
        print(f"✗ {model_name}: No saved results found")

print("=" * 80)
print(f"Summary: {skipped_count} in memory, {loaded_count} loaded from disk, {len(model_names) - skipped_count - loaded_count} missing")
print(f"Total available: {len([m for m in model_names if m in all_results and all_results[m] is not None])}/{len(model_names)} models")
print("=" * 80)


#### 5.2 Create Comparison DataFrame

In [ ]:
print("\n" + "="*80)
print("RESULTS COMPARISON")
print("="*80)

# Create comparison dataframes
overall_comparison_data = []
bin_comparison_data = []
available_models = []
missing_models = []

for model_name in ['nt5-small', 'flan-t5-base', 'indicbart', 'flan-t5-large', 'conflibert-poisson', 'mt5-base']:
    if model_name in all_results and all_results[model_name] is not None:
        metrics = all_results[model_name]
        
        # Handle both old format (flat dict) and new format (with 'overall' and 'bins')
        if 'overall' in metrics:
            overall_metrics = metrics['overall']
            bin_metrics = metrics.get('bins', {})
        else:
            # Legacy format - extract only overall metrics
            overall_metrics = {k: v for k, v in metrics.items() if k != 'bins'}
            bin_metrics = metrics.get('bins', {})
        
        # Overall metrics row
        overall_row = {'Model': model_name}
        overall_row.update(overall_metrics)
        overall_comparison_data.append(overall_row)
        
        # Per-bin metrics rows
        for bin_label in ['0', '1', '2', '3-5', '6+']:
            if bin_label in bin_metrics:
                bin_row = {
                    'Model': model_name,
                    'Bin': bin_label
                }
                bin_row.update(bin_metrics[bin_label])
                bin_comparison_data.append(bin_row)
        
        available_models.append(model_name)
    else:
        missing_models.append(model_name)

print(f"\nComparing {len(available_models)} models: {', '.join(available_models)}")
if missing_models:
    print(f"⚠️  Missing results for: {', '.join(missing_models)}")
print()

if len(overall_comparison_data) == 0:
    print("❌ No model results available for comparison!")
    print("   Please run at least one model training section first.")
else:
    # Overall metrics dataframe
    overall_df = pd.DataFrame(overall_comparison_data)
    
    # Display overall results table
    print("\n### Overall Model Comparison ###\n")
    display_cols = ['Model', 'mae', 'rmse', 'mdae', 'exact_match', 'within_1', 'within_2', 'nonzero_mae', 'extraction_rate']
    # Only show columns that exist
    display_cols = [col for col in display_cols if col in overall_df.columns]
    print(overall_df[display_cols].to_string(index=False))
    
    # Save overall metrics
    save_dataframe_csv(overall_df, 'death_counts_metrics_combined_overall.csv', task_name=TASK_NAME)
    print(f"\n✅ Overall metrics saved to {get_task_results_dir(TASK_NAME)}/death_counts_metrics_combined_overall.csv")
    
    # Per-bin metrics dataframe
    if len(bin_comparison_data) > 0:
        bins_df = pd.DataFrame(bin_comparison_data)
        
        # Display per-bin table (grouped by model)
        print("\n### Per-Bin Metrics (by Model) ###\n")
        bin_display_cols = ['Model', 'Bin', 'mae', 'rmse', 'exact_match', 'within_1', 'within_2', 'n']
        bin_display_cols = [col for col in bin_display_cols if col in bins_df.columns]
        print(bins_df[bin_display_cols].to_string(index=False))
        
        # Also show pivot table by bin for easier comparison
        print("\n### Per-Bin Metrics (by Bin) ###\n")
        for bin_label in ['0', '1', '2', '3-5', '6+']:
            bin_data = bins_df[bins_df['Bin'] == bin_label]
            if len(bin_data) > 0:
                print(f"\nBin: {bin_label}")
                print(bin_data[['Model', 'mae', 'rmse', 'exact_match', 'within_1', 'within_2', 'n']].to_string(index=False))
        
        # Save per-bin metrics
        save_dataframe_csv(bins_df, 'death_counts_metrics_combined_bins.csv', task_name=TASK_NAME)
        print(f"\n✅ Per-bin metrics saved to {get_task_results_dir(TASK_NAME)}/death_counts_metrics_combined_bins.csv")


#### 5.3 Visualize Model Comparison

In [ ]:
# Check if overall_df exists from previous cell
if 'overall_df' not in locals() or len(overall_comparison_data) == 0:
    print("⚠️  No results available for visualization.")
    print("   Please run at least one model training section first.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Use all available models (no filtering needed now)
    valid_results = overall_df
    
    # MAE comparison
    axes[0].bar(valid_results['Model'], valid_results['mae'], color='steelblue', alpha=0.7)
    axes[0].set_ylabel('MAE')
    axes[0].set_title('Mean Absolute Error')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(axis='y', alpha=0.3)
    
    # Exact Match comparison
    axes[1].bar(valid_results['Model'], valid_results['exact_match'], color='coral', alpha=0.7)
    axes[1].set_ylabel('Exact Match Accuracy')
    axes[1].set_title('Exact Match Performance')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(axis='y', alpha=0.3)
    
    # Within-1 comparison
    axes[2].bar(valid_results['Model'], valid_results['within_1'], color='green', alpha=0.7)
    axes[2].set_ylabel('Within-1 Accuracy')
    axes[2].set_title('Within-1 Performance')
    axes[2].tick_params(axis='x', rotation=45)
    axes[2].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('figures/model_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Comparison plot saved to figures/model_comparison.png")
    print(f"   Models plotted: {', '.join(valid_results['Model'].tolist())}")


#### 5.4 Save All Predictions

In [ ]:
# Save all predictions (combined)
available_predictions = {model: preds for model, preds in all_predictions.items() 
                         if preds is not None}

if len(available_predictions) == 0:
    print("❌ No predictions available to save!")
    print("   Please run at least one model training section first.")
else:
    # Get true labels from any available model
    first_model = list(available_predictions.keys())[0]
    true_labels = available_predictions[first_model]['labels']

    # Resolve identifiers and summaries with graceful fallback
    incident_numbers = available_predictions[first_model].get('incident_number')
    incident_summaries = available_predictions[first_model].get('incident_summary')

    if incident_numbers is None and 'incident_number' in test_df.columns:
        incident_numbers = test_df['incident_number'].values
    if incident_summaries is None and 'incident_summary' in test_df.columns:
        incident_summaries = test_df['incident_summary'].values
    if incident_numbers is None:
        incident_numbers = np.arange(len(true_labels)).astype(str)
    if incident_summaries is None:
        incident_summaries = [''] * len(true_labels)

    # Create combined dataframe with identifiers first
    combined_columns = {
        'incident_number': incident_numbers,
        'incident_summary': incident_summaries,
        'true_label': true_labels,
    }
    combined_columns.update({
        f'{model}_pred': available_predictions[model]['predictions']
        for model in available_predictions
    })
    all_predictions_df = pd.DataFrame(combined_columns)

    # Save using file_io utilities
    save_dataframe_csv(all_predictions_df, 'death_counts_predictions_combined.csv', task_name=TASK_NAME)
    print(f"✅ All predictions saved to {get_task_results_dir(TASK_NAME)}/death_counts_predictions_combined.csv")
    print(f"   Predictions shape: {all_predictions_df.shape}")
    print(f"   Models included: {', '.join(available_predictions.keys())}")
    print(f"\nFirst 10 rows:")
    display(all_predictions_df.head(10))
